In [0]:
%run ../common/config


In [0]:
print(env_catalog)

In [0]:
env_schema="bronze"
#print(schema)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {env_schema}")
spark.sql(f"USE SCHEMA {env_schema}")


In [0]:
file_name="providers.csv"
github_url = (
    "https://raw.githubusercontent.com/v889/Healthcare-Claims-Intelligence-Platform/refs/heads/main/data/providers/providers.csv"
)


In [0]:
import pandas as pd

url = "https://raw.githubusercontent.com/v889/Healthcare-Claims-Intelligence-Platform/main/data/providers/providers.csv"

pdf = pd.read_csv(url)

display(pdf.head())

In [0]:
providers_df = spark.createDataFrame(pdf)

display(providers_df)

In [0]:
providers_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, lit

providers_df = (
    providers_df
    .withColumn("source_file", lit(file_name))
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_system", lit("github"))
)

In [0]:
providers_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.bronze.providers"
    )

In [0]:
source_count = providers_df.count()

target_count = spark.sql(
    f"""
    SELECT COUNT(*)
    FROM {catalog}.bronze.providers
    """
).collect()[0][0]

print(f"Source Count : {source_count}")
print(f"Target Count : {target_count}")

In [0]:
spark.sql(f"""
INSERT INTO {env_catalog}.metadata.processed_files
VALUES
(
 '{file_name}',
 'github',
 current_timestamp()
)
""")